# Ensemble weights: global, size-aware, and Faster R-CNN as a score-only vote

`ensemble_temporal.ipynb` fused all members with **equal weights**. Its best was Person B's 7 models + RetinaNet (WBF IoU 0.65, val mAP 0.2998). Adding Faster R-CNN raised AP_small but lowered mAP and AP75, which suggests its boxes are less precise and pull the fused boxes off target.

Here, on the cached member predictions (CPU only, no model is re-run):

| Scheme | What changes |
|---|---|
| plain | equal weights (reference) |
| global, p | model weight `w = (mAP_model / mean mAP) ^ p`, from each model's val mAP |
| size-aware, p | weight per model **and box size** (small < 32², medium < 96², large, in original pixels): `w = (AP_size,model / mean AP_size) ^ p`. YOLO at 1280 is best on small objects, RT-DETR on large ones |
| Faster R-CNN vote | fuse the other models, then Faster R-CNN only changes the **scores** of fused boxes it agrees with (IoU ≥ 0.55), never their coordinates; optionally its unmatched boxes are added with a low score |

WBF multiplies every box's score by its weight, and that product sets both the fused score and how much the box pulls the fused position, so a per-box weight is applied by scaling scores before `fuse`.

**Overfitting check:** each scheme has a single tuned number (p or the vote weight), not one weight per model; the best scheme is also compared clip by clip against plain.

**Run after** `ensemble_temporal.ipynb`. About 30–45 min of CPU. Outputs: `runs/ensemble/weights_results.csv`, `weights_best.json`, the best fusion on every val and test frame (`fused_best_valall.pkl`, `fused_best_test.pkl`, used by `temporal_v2.ipynb`), and `submissions/W_best_weighted.csv`.

## Setup

In [1]:
import json
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path.cwd().parent            # this notebook lives in src/experiments/
ROOT = SRC.parent
sys.path.insert(0, str(SRC))
from common.evaluate import evaluate_detections
from common.pipeline import fuse, to_submission_rows
from common.submission import write_submission

DATA_ROOT = ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026"
GT_FILE = ROOT / "data_processed" / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
RUNS = ROOT / "runs"
ENS = RUNS / "ensemble"

val_meta = json.loads((DATA_ROOT / "val" / "annotations_coco.json").read_text())["images"]
VAL = [{"image_id": im["id"], "video": im["video"], "frame": im["frame_id"], "shape": (im["height"], im["width"])}
       for im in val_meta]
test = pd.read_csv(TEST_CSV)
TEST = [{"image_id": k, "video": c, "frame": int(f), "shape": (h, w)}
        for k, c, f, w, h in zip(test["image_id"], test["clip"], test["frame"], test["width"], test["height"])]
LABELLED = sorted(im["id"] for im in json.loads(GT_FILE.read_text())["images"])
VAL_BY_ID = {f["image_id"]: f for f in VAL}
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]

final_b = json.loads((RUNS / "pipeline" / "final_config.json").read_text())
B_MEMBERS = list(final_b["members"])
FRCNN, RETINA = "frcnn_baseline_ms", "retinanet_baseline_ms"
WEAK = ["rtdetr_960", "yolo_final_960"]          # the two weakest members on val (mAP ~0.19-0.23 with flip)
ALL = B_MEMBERS + [FRCNN, RETINA]
preds = {n: pickle.loads((ENS / f"valall_{n}.pkl").read_bytes()) for n in ALL}
print(f"{len(ALL)} members loaded | {len(LABELLED)} labelled val frames")

9 members loaded | 359 labelled val frames


## 1. Each member's AP by object size

The size-aware weights come from this table (labelled val frames).

In [2]:
def to_dets(boxes_by_id, ids):
    dets = []
    for i in ids:
        xyxy, s, c = boxes_by_id[i]
        dets += [{"image_id": i, "category_id": int(k) + 1, "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                  "score": float(v)} for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    return dets


member_ap = pd.DataFrame({n: evaluate_detections(GT_FILE, to_dets(preds[n], LABELLED), image_ids=LABELLED) for n in ALL}).T
member_ap[COLS].round(4)

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR
yolo_low_scale_960,0.2206,0.4145,0.2067,0.1077,0.1772,0.4036,0.3562
yolo_baseline_1280,0.2204,0.4343,0.1994,0.1527,0.2145,0.3479,0.3683
yolo_final_960,0.1951,0.3778,0.1814,0.1056,0.1748,0.3305,0.3236
yolo_final_1280,0.1984,0.3988,0.1747,0.1097,0.1961,0.3274,0.3271
rtdetr_960,0.2341,0.4480,0.2173,0.0912,0.1975,0.4114,0.3899
rtdetr_final_960,0.2486,0.4774,0.2260,0.1103,0.2182,0.4243,0.4050
rtdetr_final_960_s025,0.2534,0.4891,0.2310,0.1089,0.2162,0.4150,0.4368
frcnn_baseline_ms,0.2192,0.4622,0.1809,0.1101,0.2017,0.4057,0.4586
retinanet_baseline_ms,0.1913,0.3753,0.1724,0.0921,0.1793,0.3535,0.3983


## 2. Weighting schemes

In [3]:
SIZES = ["AP_small", "AP_medium", "AP_large"]


def size_bin(xyxy):
    area = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    return np.digitize(area, [32 ** 2, 96 ** 2])   # 0 small, 1 medium, 2 large (COCO thresholds)


def weights_for(members, scheme, p):
    # {member: array of 3 weights (small, medium, large)}
    ap = member_ap.loc[members]
    if scheme == "plain" or p == 0:
        return {n: np.ones(3) for n in members}
    if scheme == "global":
        w = (ap["mAP"] / ap["mAP"].mean()) ** p
        return {n: np.full(3, w[n]) for n in members}
    rel = (ap[SIZES] / ap[SIZES].mean()).fillna(1.0)          # a size with no ground truth (AP undefined): weight 1
    return {n: np.maximum(rel.loc[n].to_numpy(), 0.05) ** p for n in members}   # no member is switched off entirely


def box_iou(a, b):
    # IoU between xyxy boxes a (N, 4) and b (M, 4)
    iw = (np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0])).clip(0)
    ih = (np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1])).clip(0)
    inter = iw * ih
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)


def frcnn_vote(fused, fr, n, beta, add):
    # Faster R-CNN as one extra voter of weight beta that changes scores only (WBF 'avg': mean over n + beta voters)
    xyxy, s, c = fused
    fx, fs, fc = fr
    new = s * n / (n + beta)
    used = np.zeros(len(fx), bool)
    for k in np.unique(c):
        a, b = np.where(c == k)[0], np.where(fc == k)[0]
        if not len(b):
            continue
        iou = box_iou(xyxy[a], fx[b])
        j = iou.argmax(1)
        hit = iou[np.arange(len(a)), j] >= 0.55
        new[a[hit]] += beta * fs[b[j[hit]]] / (n + beta)
        used[b[(iou >= 0.55).any(0)]] = True
    if add and (~used).any():
        xyxy = np.concatenate([xyxy, fx[~used]])
        new = np.concatenate([new, beta * fs[~used] / (n + beta)])
        c = np.concatenate([c, fc[~used]])
    order = np.argsort(-new)[:500]
    return xyxy[order], new[order].astype(np.float32), c[order]


def fuse_frame(cfg, preds, image_id, shape):
    w = weights_for(cfg["members"], cfg["scheme"], cfg["p"])
    per = []
    for n in cfg["members"]:
        xyxy, s, c = preds[n][image_id]
        per.append((xyxy, s * w[n][size_bin(xyxy)], c))
    fused = fuse(per, shape, iou=cfg["iou"])
    if cfg.get("vote"):
        fused = frcnn_vote(fused, preds[FRCNN][image_id], len(cfg["members"]), **cfg["vote"])
    return fused

## 3. Grid on the labelled val frames

Member sets: B's 7 + RetinaNet (the best plain set), the same + Faster R-CNN, and both without the two weakest members. Every scheme at WBF IoU 0.65 (the best in `ensemble_temporal.ipynb`).

In [4]:
QUICK = True   # True: only the B7+Ret set (plain / global / size-aware), no Faster R-CNN vote, no val fusion in section 5.
               # False: the full grid below (tomorrow), and section 5 also fuses every val frame for temporal_v2.ipynb.
SETS = {
    "B7+Ret": B_MEMBERS + [RETINA],
    "B7+Ret+FRCNN": B_MEMBERS + [RETINA, FRCNN],
    "B7+Ret -weak": [n for n in B_MEMBERS if n not in WEAK] + [RETINA],
    "B7+Ret+FRCNN -weak": [n for n in B_MEMBERS if n not in WEAK] + [RETINA, FRCNN],
}
if QUICK:
    SETS = {"B7+Ret": SETS["B7+Ret"]}
CONFIGS = {}
for set_name, members in SETS.items():
    CONFIGS[f"{set_name} | plain"] = dict(members=members, scheme="plain", p=0)
    for scheme in ("global", "size"):
        for p in (1, 2, 4):
            CONFIGS[f"{set_name} | {scheme} p={p}"] = dict(members=members, scheme=scheme, p=p)
for beta in ((0.5, 1.0) if not QUICK else ()):
    for add in (False, True):
        CONFIGS[f"B7+Ret | plain + FRCNN vote {beta}{' +add' if add else ''}"] = dict(
            members=SETS["B7+Ret"], scheme="plain", p=0, vote=dict(beta=beta, add=add))
for cfg in CONFIGS.values():
    cfg["iou"] = 0.65

rows = {}
for label, cfg in CONFIGS.items():
    t0 = time.time()
    fused = {i: fuse_frame(cfg, preds, i, VAL_BY_ID[i]["shape"]) for i in LABELLED}
    rows[label] = evaluate_detections(GT_FILE, to_dets(fused, LABELLED), image_ids=LABELLED)
    print(f"{label:45s} mAP {rows[label]['mAP']:.4f}  AP_small {rows[label]['AP_small']:.4f}  ({time.time() - t0:.0f} s)", flush=True)

grid = pd.DataFrame(rows).T[COLS]
ref = grid.loc["B7+Ret | plain", "mAP"]
grid["delta vs B7+Ret plain"] = grid["mAP"] - ref
grid.to_csv(ENS / "weights_results.csv")
grid.sort_values("mAP", ascending=False).round(4)

B7+Ret | plain                                mAP 0.2998  AP_small 0.1435  (51 s)
B7+Ret | global p=1                           mAP 0.2999  AP_small 0.1433  (50 s)
B7+Ret | global p=2                           mAP 0.3011  AP_small 0.1433  (51 s)
B7+Ret | global p=4                           mAP 0.2992  AP_small 0.1425  (51 s)
B7+Ret | size p=1                             mAP 0.3016  AP_small 0.1455  (52 s)
B7+Ret | size p=2                             mAP 0.3027  AP_small 0.1476  (52 s)
B7+Ret | size p=4                             mAP 0.3025  AP_small 0.1506  (51 s)


,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR,delta vs B7+Ret plain
B7+Ret | size p=2,0.3027,0.5553,0.2954,0.1476,0.2593,0.4986,0.5216,0.0029
B7+Ret | size p=4,0.3025,0.5571,0.2937,0.1506,0.2576,0.4990,0.5209,0.0027
B7+Ret | size p=1,0.3016,0.5531,0.2941,0.1455,0.2584,0.4992,0.5210,0.0018
B7+Ret | global p=2,0.3011,0.5527,0.2940,0.1433,0.2576,0.4985,0.5199,0.0013
B7+Ret | global p=1,0.2999,0.5510,0.2927,0.1433,0.2580,0.4971,0.5213,0.0001
B7+Ret | plain,0.2998,0.5494,0.2914,0.1435,0.2582,0.4972,0.5208,0.0000
B7+Ret | global p=4,0.2992,0.5511,0.2902,0.1425,0.2546,0.4966,0.5154,-0.0006


**Observations:** _fill in_

## 4. Is the best scheme better on most clips, or on one?

mAP per val clip, best scheme vs plain B7+Ret.

In [5]:
BEST = grid["mAP"].idxmax()
best_cfg = CONFIGS[BEST]
print("best:", BEST)
clip_of = {f["image_id"]: f["video"] for f in VAL}
per_clip = []
fused_ref = {i: fuse_frame(CONFIGS["B7+Ret | plain"], preds, i, VAL_BY_ID[i]["shape"]) for i in LABELLED}
fused_best = {i: fuse_frame(best_cfg, preds, i, VAL_BY_ID[i]["shape"]) for i in LABELLED}
for clip in sorted(set(clip_of.values())):
    ids = [i for i in LABELLED if clip_of[i] == clip]
    per_clip.append({"clip": clip, "frames": len(ids),
                     "plain": evaluate_detections(GT_FILE, to_dets(fused_ref, ids), image_ids=ids)["mAP"],
                     "best": evaluate_detections(GT_FILE, to_dets(fused_best, ids), image_ids=ids)["mAP"]})
per_clip = pd.DataFrame(per_clip).assign(delta=lambda t: t["best"] - t["plain"])
print(f"best scheme better on {(per_clip.delta > 0).sum()} of {len(per_clip)} clips")
per_clip.round(4)

best: B7+Ret | size p=2
best scheme better on 5 of 7 clips


,clip,frames,plain,best,delta
0,uav0000086_00000_v,58,0.3124,0.3157,0.0033
1,uav0000117_02622_v,44,0.3163,0.3166,0.0004
2,uav0000137_00458_v,30,0.3448,0.3460,0.0012
3,uav0000182_00000_v,46,0.2346,0.2370,0.0023
4,uav0000268_05773_v,123,0.1978,0.2000,0.0022
5,uav0000305_00000_v,23,0.4378,0.4270,-0.0108
6,uav0000339_00001_v,35,0.3148,0.3146,-0.0001


**Observations:** _fill in_

## 5. Best scheme on every val and test frame

With `QUICK = True` only the test frames are fused (enough for the submission). Saved as flat detection lists with clip and frame number (`fused_best_valall.pkl`, `fused_best_test.pkl`): `temporal_v2.ipynb` uses them as its ensemble input. Also writes `submissions/W_best_weighted.csv`. Needs the test predictions from `ensemble_temporal.ipynb` section 5.

In [6]:
def fuse_all(split, frames, preds_split):
    t0 = time.time()
    dets = []
    for f in frames:
        xyxy, s, c = fuse_frame(best_cfg, preds_split, f["image_id"], f["shape"])
        dets += [{"image_id": f["image_id"], "video": f["video"], "frame": f["frame"], "category_id": int(k) + 1,
                  "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(v)}
                 for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    (ENS / f"fused_best_{split}.pkl").write_bytes(pickle.dumps(dets))
    print(f"{split}: fused {len(frames)} frames in {(time.time() - t0) / 60:.1f} min")
    return dets


needed = set(best_cfg["members"]) | ({FRCNN} if best_cfg.get("vote") else set())
(ENS / "weights_best.json").write_text(json.dumps({"label": BEST, **best_cfg}, indent=1))
if not QUICK:   # every val frame, for temporal_v2.ipynb
    _ = fuse_all("valall", VAL, preds)
del preds
test_preds = {n: pickle.loads((ENS / f"test_{n}.pkl").read_bytes()) for n in needed}
test_dets = fuse_all("test", TEST, test_preds)
rows_by_id = {}
for d in test_dets:
    x, y, w, h = d["bbox"]
    rows_by_id.setdefault(d["image_id"], []).append((d["category_id"] - 1, d["score"], x, y, w, h))
sub = write_submission(rows_by_id, TEST_CSV, ENS / "submissions" / "W_best_weighted.csv")
print(f"wrote W_best_weighted.csv: {len(sub)} rows")

test: fused 3811 frames in 3.7 min
wrote W_best_weighted.csv: 3811 rows
